In [ ]:
# zelle 1

In [ ]:
from pathlib import Path
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml


def find_project_root(start: Path) -> Path:
    """
    Suche vom aktuellen Ordner aus nach oben nach einem
    Projektordner, der src/walinet enthält.
    """
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (candidate / "src" / "walinet").is_dir():
            return candidate

    raise FileNotFoundError(
        "WALINET project root not found. "
        "Start the notebook somewhere inside the repository "
        "or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = find_project_root(
    Path.cwd()
)

src_dir = PROJECT_ROOT / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(
        0,
        str(src_dir),
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src directory:", src_dir)

In [ ]:
TRAIN_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "Training"
    / "train_7T.yaml"
)

SIMULATION_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "Simulation"
    / "7T_on_the_fly.yaml"
)

print("Training config:")
print(TRAIN_CONFIG_PATH)
print("Exists:", TRAIN_CONFIG_PATH.is_file())

print()

print("Simulation config:")
print(SIMULATION_CONFIG_PATH)
print("Exists:", SIMULATION_CONFIG_PATH.is_file())

assert TRAIN_CONFIG_PATH.is_file(), (
    "Training config not found:\n"
    f"{TRAIN_CONFIG_PATH}"
)

assert SIMULATION_CONFIG_PATH.is_file(), (
    "Simulation config not found:\n"
    f"{SIMULATION_CONFIG_PATH}"
)

In [ ]:
from walinet.config.build import (
    build_config,
)
from walinet.config.build_simulation import (
    build_simulation_config,
)


def load_yaml(
    path: Path,
) -> dict:
    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        raw = yaml.safe_load(
            file
        )

    if not isinstance(
        raw,
        dict,
    ):
        raise TypeError(
            "Expected a YAML mapping in:\n"
            f"{path}\n"
            f"Found: {type(raw)}"
        )

    return raw


train_raw = load_yaml(
    TRAIN_CONFIG_PATH
)

simulation_raw = load_yaml(
    SIMULATION_CONFIG_PATH
)

train_cfg = build_config(
    train_raw,
    config_dir=TRAIN_CONFIG_PATH.parent,
)

simulation_cfg = (
    build_simulation_config(
        simulation_raw,
        config_dir=(
            SIMULATION_CONFIG_PATH.parent
        ),
    )
)

print(
    "Both configs loaded and "
    "validated successfully."
)

In [ ]:
print("TRAINING CONFIG")
print("=" * 60)

print(
    "Run name:",
    train_cfg.run.name,
)

print(
    "Data source:",
    train_cfg.data.source,
)

print(
    "Base directory:",
    train_cfg.data.base_dir,
)

print(
    "Train subjects:",
    len(
        train_cfg.data.train_subjects
    ),
)

print(
    "Validation subjects:",
    len(
        train_cfg.data.val_subjects
    ),
)

print(
    "Normalization:",
    train_cfg.data.normalization,
)

print(
    "Training batch size:",
    train_cfg.training.batch_size,
)

print(
    "Validation spectra:",
    train_cfg.validation.n_spectra,
)

print(
    "Validation seed:",
    train_cfg.validation.seed,
)


print()
print("SIMULATION CONFIG")
print("=" * 60)

print(
    "Version:",
    simulation_cfg.version,
)

print(
    "Bandwidth [Hz]:",
    simulation_cfg
    .acquisition
    .bandwidth_hz,
)

print(
    "Target timepoints:",
    simulation_cfg
    .acquisition
    .n_timepoints,
)

print(
    "NMR frequency [Hz]:",
    simulation_cfg
    .acquisition
    .nmr_frequency_hz,
)

print(
    "Subject mixing:",
    simulation_cfg
    .subject_sampling
    .mixing,
)

print(
    "Lipid projection enabled:",
    simulation_cfg
    .lipid_projection
    .enabled,
)

print(
    "Basis config:",
    simulation_cfg
    .basis
    .config,
)

print(
    "Metabolite config:",
    simulation_cfg
    .metabolites
    .config,
)

In [ ]:
assert (
    train_cfg.data.source
    == "on_the_fly"
)

assert (
    train_cfg.data.on_the_fly
    is not None
)

resource_template = (
    train_cfg
    .data
    .on_the_fly
    .resources
    .filename
)

resource_version = (
    train_cfg
    .data
    .on_the_fly
    .resources
    .version
)

resource_relative_path = Path(
    resource_template.format(
        version=resource_version,
    )
)

base_dir = Path(
    train_cfg.data.base_dir
)

print(
    "Relative resource path:",
    resource_relative_path,
)

print(
    "Base directory:",
    base_dir,
)

In [ ]:
def collect_resource_paths(
    subjects: list[str],
) -> pd.DataFrame:
    rows = []

    for subject in subjects:
        path = (
            base_dir
            / subject
            / resource_relative_path
        )

        rows.append(
            {
                "subject": subject,
                "resource_path": str(path),
                "exists": path.is_file(),
                "size_mb": (
                    path.stat().st_size
                    / 1024**2
                    if path.is_file()
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


train_paths = collect_resource_paths(
    train_cfg.data.train_subjects
)

validation_paths = (
    collect_resource_paths(
        train_cfg.data.val_subjects
    )
)

print("Training resources")
display(train_paths)

print("Validation resources")
display(validation_paths)


missing_train = train_paths.loc[
    ~train_paths["exists"],
    "subject",
].tolist()

missing_validation = (
    validation_paths.loc[
        ~validation_paths["exists"],
        "subject",
    ].tolist()
)

assert not missing_train, (
    "Missing training resources:\n"
    f"{missing_train}"
)

assert not missing_validation, (
    "Missing validation resources:\n"
    f"{missing_validation}"
)

print(
    "All expected resource files exist."
)

In [ ]:
from walinet.training_data.simulation_resources import (
    SimulationPool,
    SimulationResources,
    build_simulation_resources,
)


resources = build_simulation_resources(
    train_cfg=train_cfg,
    simulation_cfg=simulation_cfg,
)

print(
    "Resource pools created successfully."
)

In [ ]:
def tensor_size_mb(
    tensor: torch.Tensor | None,
) -> float:
    if tensor is None:
        return 0.0

    return (
        tensor.numel()
        * tensor.element_size()
        / 1024**2
    )

In [ ]:
def summarize_pool(
    name: str,
    pool: SimulationPool,
) -> None:
    print(name)
    print("=" * 60)

    print(
        "Subjects:",
        pool.n_subjects,
    )

    print(
        "Subject names:",
        pool.subject_names,
    )

    print(
        "Water spectra:",
        tuple(
            pool.water_spectra.shape
        ),
        pool.water_spectra.dtype,
    )

    print(
        "Lipid spectra:",
        tuple(
            pool.lipid_spectra.shape
        ),
        pool.lipid_spectra.dtype,
    )

    print(
        "Water offsets:",
        tuple(
            pool.water_offsets.shape
        ),
    )

    print(
        "Lipid offsets:",
        tuple(
            pool.lipid_offsets.shape
        ),
    )

    print(
        "Water counts per subject:",
        pool.water_counts.tolist(),
    )

    print(
        "Lipid counts per subject:",
        pool.lipid_counts.tolist(),
    )

    print(
        "Native lengths:",
        pool.native_lengths.tolist(),
    )

    print(
        "Bandwidth [Hz]:",
        pool.bandwidth_hz,
    )

    print(
        "Target spectral points:",
        pool.n_timepoints,
    )

    print(
        "Device:",
        pool.device,
    )

    print(
        "Water memory [MB]:",
        round(
            tensor_size_mb(
                pool.water_spectra
            ),
            2,
        ),
    )

    print(
        "Lipid memory [MB]:",
        round(
            tensor_size_mb(
                pool.lipid_spectra
            ),
            2,
        ),
    )

    print(
        "Projection memory [MB]:",
        round(
            tensor_size_mb(
                pool.lipid_projection_operators
            ),
            2,
        ),
    )

    total_memory = (
        tensor_size_mb(
            pool.water_spectra
        )
        + tensor_size_mb(
            pool.lipid_spectra
        )
        + tensor_size_mb(
            pool.lipid_projection_operators
        )
    )

    print(
        "Total main tensor memory [MB]:",
        round(
            total_memory,
            2,
        ),
    )

    print()


summarize_pool(
    "TRAIN POOL",
    resources.train,
)

summarize_pool(
    "VALIDATION POOL",
    resources.validation,
)

In [ ]:
def validate_pool(
    pool: SimulationPool,
    expected_subjects: list[str],
) -> None:
    target_t = (
        simulation_cfg
        .acquisition
        .n_timepoints
    )

    expected_bandwidth = (
        simulation_cfg
        .acquisition
        .bandwidth_hz
    )

    assert (
        pool.subject_names
        == tuple(expected_subjects)
    )

    assert (
        pool.n_subjects
        == len(expected_subjects)
    )

    assert pool.water_spectra.ndim == 2
    assert pool.lipid_spectra.ndim == 2

    assert (
        pool.water_spectra.shape[1]
        == target_t
    )

    assert (
        pool.lipid_spectra.shape[1]
        == target_t
    )

    assert (
        pool.water_spectra.dtype
        == torch.complex64
    )

    assert (
        pool.lipid_spectra.dtype
        == torch.complex64
    )

    assert (
        pool.water_offsets.dtype
        == torch.int64
    )

    assert (
        pool.lipid_offsets.dtype
        == torch.int64
    )

    assert (
        pool.native_lengths.dtype
        == torch.int64
    )

    assert (
        pool.water_offsets.shape
        == (pool.n_subjects + 1,)
    )

    assert (
        pool.lipid_offsets.shape
        == (pool.n_subjects + 1,)
    )

    assert (
        pool.native_lengths.shape
        == (pool.n_subjects,)
    )

    assert (
        int(
            pool.water_offsets[0]
        )
        == 0
    )

    assert (
        int(
            pool.lipid_offsets[0]
        )
        == 0
    )

    assert torch.all(
        pool.water_offsets[1:]
        >= pool.water_offsets[:-1]
    )

    assert torch.all(
        pool.lipid_offsets[1:]
        >= pool.lipid_offsets[:-1]
    )

    assert (
        int(
            pool.water_offsets[-1]
        )
        == pool.n_water_spectra
    )

    assert (
        int(
            pool.lipid_offsets[-1]
        )
        == pool.n_lipid_spectra
    )

    assert torch.all(
        pool.water_counts > 0
    )

    assert torch.all(
        pool.lipid_counts > 0
    )

    assert torch.all(
        pool.native_lengths > 0
    )

    assert torch.isfinite(
        pool.water_spectra.real
    ).all()

    assert torch.isfinite(
        pool.water_spectra.imag
    ).all()

    assert torch.isfinite(
        pool.lipid_spectra.real
    ).all()

    assert torch.isfinite(
        pool.lipid_spectra.imag
    ).all()

    assert not torch.any(
        torch.all(
            pool.water_spectra == 0,
            dim=-1,
        )
    )

    assert not torch.any(
        torch.all(
            pool.lipid_spectra == 0,
            dim=-1,
        )
    )

    assert (
        pool.n_timepoints
        == target_t
    )

    assert np.isclose(
        pool.bandwidth_hz,
        expected_bandwidth,
    )

    assert (
        pool.device
        == pool.water_spectra.device
    )

    assert (
        pool.lipid_spectra.device
        == pool.device
    )

    assert (
        pool.water_offsets.device
        == pool.device
    )

    assert (
        pool.lipid_offsets.device
        == pool.device
    )

    assert (
        pool.native_lengths.device
        == pool.device
    )

    if (
        simulation_cfg
        .lipid_projection
        .enabled
    ):
        assert (
            pool
            .lipid_projection_operators
            is not None
        )

        assert (
            pool
            .lipid_projection_operators
            .shape
            == (
                pool.n_subjects,
                target_t,
                target_t,
            )
        )

        assert (
            pool
            .lipid_projection_operators
            .dtype
            == torch.complex64
        )

        assert (
            pool
            .lipid_projection_operators
            .device
            == pool.device
        )

        assert torch.isfinite(
            pool
            .lipid_projection_operators
            .real
        ).all()

        assert torch.isfinite(
            pool
            .lipid_projection_operators
            .imag
        ).all()

    else:
        assert (
            pool
            .lipid_projection_operators
            is None
        )

In [ ]:
validate_pool(
    resources.train,
    train_cfg.data.train_subjects,
)

validate_pool(
    resources.validation,
    train_cfg.data.val_subjects,
)


overlap = (
    set(
        resources
        .train
        .subject_names
    )
    & set(
        resources
        .validation
        .subject_names
    )
)

assert not overlap, (
    "Train/validation overlap found:\n"
    f"{sorted(overlap)}"
)

print(
    "All structural pool checks passed."
)

In [ ]:
def subject_table(
    pool: SimulationPool,
) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "subject_index": (
                np.arange(
                    pool.n_subjects
                )
            ),
            "subject": (
                pool.subject_names
            ),
            "water_fids": (
                pool
                .water_counts
                .cpu()
                .numpy()
            ),
            "lipid_fids": (
                pool
                .lipid_counts
                .cpu()
                .numpy()
            ),
            "native_timepoints": (
                pool
                .native_lengths
                .cpu()
                .numpy()
            ),
        }
    )


print("Training subjects")

display(
    subject_table(
        resources.train
    )
)


print("Validation subjects")

display(
    subject_table(
        resources.validation
    )
)

In [ ]:
POOL_TO_INSPECT = (
    resources.train
)

SUBJECT_INDEX = 0


subject_name = (
    POOL_TO_INSPECT
    .subject_name(
        SUBJECT_INDEX
    )
)

water_subject = (
    POOL_TO_INSPECT
    .water_for_subject(
        SUBJECT_INDEX
    )
)

lipid_subject = (
    POOL_TO_INSPECT
    .lipids_for_subject(
        SUBJECT_INDEX
    )
)


print(
    "Subject:",
    subject_name,
)

print(
    "Water subset:",
    tuple(
        water_subject.shape
    ),
)

print(
    "Lipid subset:",
    tuple(
        lipid_subject.shape
    ),
)

In [ ]:
water_start = int(
    POOL_TO_INSPECT
    .water_offsets[
        SUBJECT_INDEX
    ]
)

water_end = int(
    POOL_TO_INSPECT
    .water_offsets[
        SUBJECT_INDEX + 1
    ]
)

lipid_start = int(
    POOL_TO_INSPECT
    .lipid_offsets[
        SUBJECT_INDEX
    ]
)

lipid_end = int(
    POOL_TO_INSPECT
    .lipid_offsets[
        SUBJECT_INDEX + 1
    ]
)


assert torch.equal(
    water_subject,
    POOL_TO_INSPECT
    .water_spectra[
        water_start:water_end
    ],
)

assert torch.equal(
    lipid_subject,
    POOL_TO_INSPECT
    .lipid_spectra[
        lipid_start:lipid_end
    ],
)

print(
    "Subject helper methods agree "
    "with the stored offsets."
)

In [ ]:
WATER_EXAMPLE_INDEX = 0

water_spectrum = (
    water_subject[
        WATER_EXAMPLE_INDEX
    ]
    .detach()
    .cpu()
    .numpy()
)

water_fid = np.fft.ifft(
    np.fft.ifftshift(
        water_spectrum
    )
)


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        water_fid
    )
)

plt.title(
    f"Water FID reconstructed from spectrum – {subject_name}"
)

plt.xlabel(
    "Timepoint"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        water_spectrum
    )
)

plt.title(
    f"Water spectrum – {subject_name}"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
LIPID_EXAMPLE_INDEX = 0

lipid_spectrum = (
    lipid_subject[
        LIPID_EXAMPLE_INDEX
    ]
    .detach()
    .cpu()
    .numpy()
)

lipid_fid = np.fft.ifft(
    np.fft.ifftshift(
        lipid_spectrum
    )
)


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        lipid_fid
    )
)

plt.title(
    f"Lipid FID reconstructed from spectrum – {subject_name}"
)

plt.xlabel(
    "Timepoint"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        lipid_spectrum
    )
)

plt.title(
    f"Lipid spectrum – {subject_name}"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
def sample_global_indices_for_subjects(
    offsets: torch.Tensor,
    subject_indices: torch.Tensor,
    *,
    generator: torch.Generator,
) -> torch.Tensor:
    """
    Wähle für jedes angegebene Subject einen zufälligen
    lokalen FID-Index und rechne ihn in einen globalen
    Pool-Index um.
    """
    counts = (
        offsets[1:]
        - offsets[:-1]
    )

    selected_counts = counts[
        subject_indices
    ]

    random_values = torch.rand(
        subject_indices.shape,
        generator=generator,
        device=subject_indices.device,
    )

    local_indices = torch.floor(
        random_values
        * selected_counts
    ).to(
        torch.int64
    )

    global_indices = (
        offsets[
            subject_indices
        ]
        + local_indices
    )

    return global_indices

In [ ]:
test_generator_1 = (
    torch.Generator(
        device="cpu"
    )
)

test_generator_1.manual_seed(
    12345
)


test_generator_2 = (
    torch.Generator(
        device="cpu"
    )
)

test_generator_2.manual_seed(
    12345
)


subject_indices = torch.tensor(
    [
        0,
        0,
        1,
        1,
        resources.train.n_subjects - 1,
    ],
    dtype=torch.int64,
)


indices_1 = (
    sample_global_indices_for_subjects(
        resources
        .train
        .water_offsets,
        subject_indices,
        generator=test_generator_1,
    )
)

indices_2 = (
    sample_global_indices_for_subjects(
        resources
        .train
        .water_offsets,
        subject_indices,
        generator=test_generator_2,
    )
)


print(
    "Subject indices:",
    subject_indices.tolist(),
)

print(
    "Sampled global water indices:",
    indices_1.tolist(),
)


assert torch.equal(
    indices_1,
    indices_2,
)


for (
    subject_index,
    global_index,
) in zip(
    subject_indices.tolist(),
    indices_1.tolist(),
):
    start = int(
        resources
        .train
        .water_offsets[
            subject_index
        ]
    )

    end = int(
        resources
        .train
        .water_offsets[
            subject_index + 1
        ]
    )

    assert (
        start
        <= global_index
        < end
    )


print(
    "Deterministic subject-aware "
    "index sampling works."
)

In [ ]:
operators = (
    resources
    .train
    .lipid_projection_operators
)


if operators is None:
    print(
        "No projection operators were loaded."
    )

    print(
        "This is expected when "
        "lipid_projection.enabled is false."
    )

else:
    print(
        "Projection operators:",
        tuple(
            operators.shape
        ),
        operators.dtype,
    )

    operator = (
        operators[
            SUBJECT_INDEX
        ]
        .cpu()
        .numpy()
    )

    print(
        "Selected operator shape:",
        operator.shape,
    )

    print(
        "Finite:",
        np.isfinite(
            operator
        ).all(),
    )

In [ ]:
from walinet.training_data.simulator import (
    SampledResources,
    SimulationResourceSampler,
)


sampler = SimulationResourceSampler(
    pool=resources.train,
    config=simulation_cfg,
)

print(
    "Sampler created successfully."
)

print(
    "Mixing mode:",
    sampler.mixing,
)

print(
    "Number of lipid spectra per simulated spectrum:",
    sampler.n_random_lipid_spectra,
)

print(
    "Device:",
    sampler.pool.device,
)

In [ ]:
generator = torch.Generator(
    device=resources.train.device
)

generator.manual_seed(
    12345
)


sampled = sampler.sample(
    batch_size=8,
    generator=generator,
)


print(
    "water_spectra:",
    sampled.water_spectra.shape,
    sampled.water_spectra.dtype,
)

print(
    "lipid_spectra:",
    sampled.lipid_spectra.shape,
    sampled.lipid_spectra.dtype,
)

print(
    "water_subject_indices:",
    sampled.water_subject_indices,
)

print(
    "lipid_subject_indices:",
    sampled.lipid_subject_indices,
)

print(
    "water_resource_indices:",
    sampled.water_resource_indices,
)

print(
    "lipid_resource_indices:",
    sampled.lipid_resource_indices,
)

In [ ]:
water_subject_names = [
    resources.train.subject_names[
        subject_index
    ]
    for subject_index in (
        sampled
        .water_subject_indices
        .cpu()
        .tolist()
    )
]

lipid_subject_names = [
    resources.train.subject_names[
        subject_index
    ]
    for subject_index in (
        sampled
        .lipid_subject_indices
        .cpu()
        .tolist()
    )
]


for batch_index, (
    water_subject,
    lipid_subject,
) in enumerate(
    zip(
        water_subject_names,
        lipid_subject_names,
    )
):
    print(
        f"{batch_index}: "
        f"water={water_subject}, "
        f"lipids={lipid_subject}"
    )

In [ ]:
generator_1 = torch.Generator(
    device=resources.train.device
)

generator_1.manual_seed(
    12345
)


generator_2 = torch.Generator(
    device=resources.train.device
)

generator_2.manual_seed(
    12345
)


sampled_1 = sampler.sample(
    batch_size=32,
    generator=generator_1,
)

sampled_2 = sampler.sample(
    batch_size=32,
    generator=generator_2,
)


assert torch.equal(
    sampled_1.water_subject_indices,
    sampled_2.water_subject_indices,
)

assert torch.equal(
    sampled_1.lipid_subject_indices,
    sampled_2.lipid_subject_indices,
)

assert torch.equal(
    sampled_1.water_resource_indices,
    sampled_2.water_resource_indices,
)

assert torch.equal(
    sampled_1.lipid_resource_indices,
    sampled_2.lipid_resource_indices,
)

assert torch.equal(
    sampled_1.water_spectra,
    sampled_2.water_spectra,
)

assert torch.equal(
    sampled_1.lipid_spectra,
    sampled_2.lipid_spectra,
)


print(
    "Sampling is fully deterministic "
    "for identical seeds."
)

In [ ]:
for batch_index in range(
    sampled.batch_size
):
    water_subject_index = int(
        sampled
        .water_subject_indices[
            batch_index
        ]
    )

    water_resource_index = int(
        sampled
        .water_resource_indices[
            batch_index
        ]
    )

    water_start = int(
        resources
        .train
        .water_offsets[
            water_subject_index
        ]
    )

    water_end = int(
        resources
        .train
        .water_offsets[
            water_subject_index + 1
        ]
    )

    assert (
        water_start
        <= water_resource_index
        < water_end
    )


    lipid_subject_index = int(
        sampled
        .lipid_subject_indices[
            batch_index
        ]
    )

    lipid_start = int(
        resources
        .train
        .lipid_offsets[
            lipid_subject_index
        ]
    )

    lipid_end = int(
        resources
        .train
        .lipid_offsets[
            lipid_subject_index + 1
        ]
    )

    lipid_indices = (
        sampled
        .lipid_resource_indices[
            batch_index
        ]
    )

    assert torch.all(
        lipid_indices
        >= lipid_start
    )

    assert torch.all(
        lipid_indices
        < lipid_end
    )


print(
    "All sampled spectra belong to "
    "their selected subjects."
)

In [ ]:
batch_index = 0

water_spectrum = (
    sampled
    .water_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

lipid_spectra = (
    sampled
    .lipid_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)


# Optional: reconstruct FIDs for inspection.
water_fid = np.fft.ifft(
    np.fft.ifftshift(
        water_spectrum
    )
)

lipid_fids = np.fft.ifft(
    np.fft.ifftshift(
        lipid_spectra,
        axes=-1,
    ),
    axis=-1,
)


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.abs(
        water_fid
    )
)

plt.title(
    "Sampled water FID reconstructed from spectrum"
)

plt.xlabel(
    "Timepoint"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


plt.figure(
    figsize=(12, 5)
)

for lipid_index in range(
    lipid_fids.shape[0]
):
    plt.plot(
        np.abs(
            lipid_fids[
                lipid_index
            ]
        ),
        alpha=0.7,
    )

plt.title(
    "Sampled lipid FIDs reconstructed from spectra"
)

plt.xlabel(
    "Timepoint"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
from walinet.training_data.simulator import (
    LipidMixture,
    mix_sampled_lipid_spectra,
)

In [ ]:
lipid_generator = torch.Generator(
    device=sampled.device
)

lipid_generator.manual_seed(
    23456
)

lipid_mixture = mix_sampled_lipid_spectra(
    sampled=sampled,
    generator=lipid_generator,
)

print(
    "mixed_spectra:",
    lipid_mixture.mixed_spectra.shape,
    lipid_mixture.mixed_spectra.dtype,
)

print(
    "weights:",
    lipid_mixture.weights.shape,
    lipid_mixture.weights.dtype,
)

print(
    "weight sums:",
    lipid_mixture.weights.sum(dim=1),
)

In [ ]:
spectrum_mixed_manually = torch.sum(
    sampled.lipid_spectra
    * lipid_mixture.weights.unsqueeze(-1),
    dim=1,
)

max_difference = torch.max(
    torch.abs(
        spectrum_mixed_manually
        - lipid_mixture.mixed_spectra
    )
)

print(
    "Maximum difference:",
    max_difference.item(),
)

assert torch.allclose(
    spectrum_mixed_manually,
    lipid_mixture.mixed_spectra,
    rtol=1e-6,
    atol=1e-6,
)

print(
    "Spectrum-domain lipid mixing is correct."
)

In [ ]:
batch_index = 0

individual_lipid_spectra = (
    sampled
    .lipid_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

mixed_lipid_spectrum = (
    lipid_mixture
    .mixed_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

plt.figure(
    figsize=(12, 5)
)

for component_index in range(
    individual_lipid_spectra.shape[0]
):
    plt.plot(
        np.abs(
            individual_lipid_spectra[
                component_index
            ]
        ),
        alpha=0.3,
    )

plt.plot(
    np.abs(
        mixed_lipid_spectrum
    ),
    linewidth=2.0,
    label="Weighted mixture",
)

plt.title(
    "Individual lipid spectra and mixed baseline"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Magnitude"
)

plt.grid(
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
from walinet.training_data.lcmodel_basis.acquisition import (
    prepare_basis_for_acquisition,
)


prepared_basis = prepare_basis_for_acquisition(
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/MetabModes/LCModelBasis/processed/walinet_7T_native_basis_v1.h5",
    target_bandwidth=2778.0,
    target_n_timepoints=558,
)

from walinet.training_data.metabolite_simulation import (
    MetaboliteSimulator,
    SimulatedMetabolites,
)


metabolite_simulator = MetaboliteSimulator(
    prepared_basis=prepared_basis,
    config=simulation_cfg,
    device=resources.train.device,
)

print(
    "Metabolite simulator created."
)

print(
    "Basis components:",
    metabolite_simulator.n_basis_components,
)

print(
    "Active metabolites:",
    metabolite_simulator
    .sampling_table
    .n_active_components,
)

print(
    "Timepoints:",
    metabolite_simulator.n_timepoints,
)

print(
    "Device:",
    metabolite_simulator.device,
)

In [ ]:
table = (
    metabolite_simulator
    .sampling_table
)

print("Active metabolite mappings")
print("=" * 60)

for (
    config_name,
    basis_name,
) in zip(
    table.active_config_names,
    table.active_basis_names,
):
    basis_index = (
        table
        .basis_names
        .index(
            basis_name
        )
    )

    print(
        f"{config_name:8s} "
        f"-> {basis_name:12s} "
        f"mean={table.means[basis_index].item():6.2f} "
        f"std={table.stds[basis_index].item():6.2f}"
    )

In [ ]:
metabolite_generator = torch.Generator(
    device=resources.train.device
)

metabolite_generator.manual_seed(
    34567
)


simulated_metabolites = (
    metabolite_simulator.simulate(
        batch_size=8,
        generator=metabolite_generator,
    )
)


print(
    "clean_fids:",
    simulated_metabolites.clean_fids.shape,
    simulated_metabolites.clean_fids.dtype,
)

print(
    "clean_spectra:",
    simulated_metabolites.clean_spectra.shape,
    simulated_metabolites.clean_spectra.dtype,
)

print(
    "concentrations:",
    simulated_metabolites.concentrations.shape,
    simulated_metabolites.concentrations.dtype,
)

print(
    "acquisition delays [s]:",
    simulated_metabolites
    .acquisition_delays_seconds,
)

print(
    "frequency shifts [Hz]:",
    simulated_metabolites
    .frequency_shifts_hz,
)

print(
    "global phases [rad]:",
    simulated_metabolites
    .global_phases_radians,
)

In [ ]:
cfg_metab = (
    simulation_cfg.metabolites
)

line_cfg = (
    cfg_metab.line_broadening
)


assert torch.all(
    torch.abs(
        simulated_metabolites
        .acquisition_delays_seconds
    )
    <= (
        cfg_metab
        .max_acquisition_delay_seconds
        + 1e-8
    )
)

assert torch.all(
    torch.abs(
        simulated_metabolites
        .frequency_shifts_hz
    )
    <= (
        cfg_metab
        .max_frequency_shift_hz
        + 1e-6
    )
)

assert torch.all(
    simulated_metabolites
    .total_broadening
    >= line_cfg.minimum
)

assert torch.all(
    simulated_metabolites
    .total_broadening
    <= line_cfg.maximum
)

assert torch.all(
    simulated_metabolites
    .gaussian_fractions
    >= line_cfg.gaussian_fraction_min
)

assert torch.all(
    simulated_metabolites
    .gaussian_fractions
    <= line_cfg.gaussian_fraction_max
)

assert torch.allclose(
    simulated_metabolites
    .gaussian_broadening
    + simulated_metabolites
    .lorentzian_broadening,
    simulated_metabolites
    .total_broadening,
    rtol=1e-6,
    atol=1e-6,
)

print(
    "All sampled parameter ranges are valid."
)

In [ ]:
disabled_mask = ~(
    metabolite_simulator
    .sampling_table
    .enabled_mask
)

disabled_concentrations = (
    simulated_metabolites
    .concentrations[
        :,
        disabled_mask,
    ]
)

assert torch.all(
    disabled_concentrations == 0
)

print(
    "All disabled basis components have "
    "zero concentration."
)

In [ ]:
generator_1 = torch.Generator(
    device=resources.train.device
)
generator_1.manual_seed(34567)

generator_2 = torch.Generator(
    device=resources.train.device
)
generator_2.manual_seed(34567)


metabolites_1 = (
    metabolite_simulator.simulate(
        batch_size=32,
        generator=generator_1,
    )
)

metabolites_2 = (
    metabolite_simulator.simulate(
        batch_size=32,
        generator=generator_2,
    )
)


assert torch.equal(
    metabolites_1.concentrations,
    metabolites_2.concentrations,
)

assert torch.equal(
    metabolites_1.clean_fids,
    metabolites_2.clean_fids,
)

assert torch.equal(
    metabolites_1.clean_spectra,
    metabolites_2.clean_spectra,
)

assert torch.equal(
    metabolites_1.acquisition_delays_seconds,
    metabolites_2.acquisition_delays_seconds,
)

assert torch.equal(
    metabolites_1.frequency_shifts_hz,
    metabolites_2.frequency_shifts_hz,
)

print(
    "Metabolite simulation is deterministic "
    "for identical seeds."
)

In [ ]:
spectra = (
    simulated_metabolites
    .clean_spectra
    .cpu()
    .numpy()
)

plt.figure(
    figsize=(12, 6)
)

for index in range(
    spectra.shape[0]
):
    plt.plot(
        np.real(
            spectra[index]
        ),
        alpha=0.7,
    )

plt.title(
    "Simulated noise-free metabolite spectra"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Real signal"
)

plt.show()

In [ ]:
from walinet.training_data.metabolite_noise import (
    SimulatedNoise,
    simulate_receiver_noise,
)


noise_generator = torch.Generator(
    device=simulated_metabolites.device
)

noise_generator.manual_seed(
    45678
)


simulated_noise = simulate_receiver_noise(
    metabolites=simulated_metabolites,
    config=simulation_cfg,
    generator=noise_generator,
)


print(
    "noise_spectra:",
    simulated_noise.noise_spectra.shape,
    simulated_noise.noise_spectra.dtype,
)

print(
    "SNR:",
    simulated_noise.snr,
)

print(
    "clean_spectrum_std:",
    simulated_noise.clean_spectrum_std,
)

print(
    "noise_scale:",
    simulated_noise.noise_scale,
)

In [ ]:
expected_noisy_metabolite_spectra = (
    simulated_metabolites.clean_spectra
    + simulated_noise.noise_spectra
)

assert (
    simulated_noise.noise_spectra.shape
    == simulated_metabolites.clean_spectra.shape
)

assert (
    simulated_noise.noise_spectra.dtype
    == simulated_metabolites.clean_spectra.dtype
)

assert (
    simulated_noise.noise_spectra.device
    == simulated_metabolites.clean_spectra.device
)

assert torch.all(
    simulated_noise.snr
    >= simulation_cfg.noise.snr_min
)

assert torch.all(
    simulated_noise.snr
    <= simulation_cfg.noise.snr_max
)

assert torch.all(
    simulated_noise.noise_scale
    > 0
)

assert torch.isfinite(
    simulated_noise.noise_spectra.real
).all()

assert torch.isfinite(
    simulated_noise.noise_spectra.imag
).all()

assert torch.isfinite(
    expected_noisy_metabolite_spectra.real
).all()

assert torch.isfinite(
    expected_noisy_metabolite_spectra.imag
).all()


# Verify the implemented scaling rule:
#
# noise_scale =
#     std(clean spectrum)
#     / 0.65
#     / SNR

expected_noise_scale = (
    simulated_noise.clean_spectrum_std
    / 0.65
    / simulated_noise.snr
)

assert torch.allclose(
    simulated_noise.noise_scale,
    expected_noise_scale,
    rtol=1e-6,
    atol=1e-6,
)


print(
    "Frequency-domain receiver-noise simulation "
    "is consistent."
)

In [ ]:
batch_index = 0

clean_spectrum = (
    simulated_metabolites
    .clean_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

noise_spectrum = (
    simulated_noise
    .noise_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

noisy_spectrum = (
    clean_spectrum
    + noise_spectrum
)


plt.figure(
    figsize=(12, 5)
)

plt.plot(
    np.real(clean_spectrum),
    label="Clean metabolites",
)

plt.plot(
    np.real(noisy_spectrum),
    label="Metabolites + receiver noise",
    alpha=0.8,
)

plt.title(
    "Frequency-domain receiver noise"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Real signal"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


plt.figure(
    figsize=(12, 4)
)

plt.plot(
    np.real(
        noise_spectrum
    )
)

plt.title(
    f"Noise spectrum, sampled SNR = "
    f"{simulated_noise.snr[batch_index].item():.2f}"
)

plt.xlabel(
    "Frequency index"
)

plt.ylabel(
    "Real noise"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
from walinet.training_data.spectrum_assembly import (
    AssembledSpectra,
    assemble_spectra,
)


assembly_generator = torch.Generator(
    device=sampled.device
)

assembly_generator.manual_seed(
    56789
)


assembled = assemble_spectra(
    sampled=sampled,
    lipid_mixture=lipid_mixture,
    metabolites=simulated_metabolites,
    noise=simulated_noise,
    pool=resources.train,
    config=simulation_cfg,
    generator=assembly_generator,
)


print(
    "Clean metabolites:",
    assembled.clean_metabolite_spectra.shape,
)

print(
    "Noisy metabolites:",
    assembled.noisy_metabolite_spectra.shape,
)

print(
    "Water:",
    assembled.water_spectra.shape,
)

print(
    "Lipids:",
    assembled.lipid_spectra.shape,
)

print(
    "Clean mixture:",
    assembled.clean_mixture_spectra.shape,
)

print(
    "Noisy mixture:",
    assembled.mixture_spectra.shape,
)

print(
    "Projected:",
    (
        None
        if assembled.projected_spectra is None
        else assembled.projected_spectra.shape
    ),
)

print(
    "Final network input:",
    assembled.input_spectra.shape,
)

print(
    "Water scaling:",
    assembled.water_scaling,
)

print(
    "Lipid scaling:",
    assembled.lipid_scaling,
)

print(
    "SNR:",
    assembled.noise.snr,
)

In [ ]:
batch_index = 0

metabolites_np = (
    assembled
    .metabolite_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

water_np = (
    assembled
    .water_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

lipids_np = (
    assembled
    .lipid_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

mixture_np = (
    assembled
    .mixture_spectra[
        batch_index
    ]
    .detach()
    .cpu()
    .numpy()
)

In [ ]:
frequency_axis_hz = np.fft.fftshift(
    np.fft.fftfreq(
        simulation_cfg.acquisition.n_timepoints,
        d=1.0 / simulation_cfg.acquisition.bandwidth_hz,
    )
)

ppm_axis = (
    prepared_basis.ppm_reference
    - frequency_axis_hz
    / prepared_basis.hz_per_ppm
)

print(
    "ppm range:",
    ppm_axis.min(),
    "to",
    ppm_axis.max(),
)

plt.figure(figsize=(12, 5))

plt.plot(
    ppm_axis,
    np.real(metabolites_np),
    label="Metabolites + noise",
)

plt.plot(
    ppm_axis,
    np.real(lipids_np),
    label="Lipids",
)

plt.xlim(7.5, 0.0)
plt.xlabel("Chemical shift [ppm]")
plt.ylabel("Real signal")
plt.title("Simulated metabolites and lipids")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    ppm_axis,
    np.real(water_np),
    label="Water",
)

plt.plot(
    ppm_axis,
    np.real(mixture_np),
    label="Complete mixture",
    alpha=0.8,
)

plt.xlim(3.5, 0.0)
plt.xlabel("Chemical shift [ppm]")
plt.ylabel("Real signal")
plt.title("Simulated water and complete spectrum")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch


sorted_indices = torch.argsort(
    simulated_noise.snr
)

positions = np.linspace(
    0,
    len(sorted_indices) - 1,
    num=min(4, len(sorted_indices)),
    dtype=int,
)

selected_indices = [
    int(sorted_indices[position].item())
    for position in positions
]


visible_mask = (
    (ppm_axis >= 0.0)
    & (ppm_axis <= 7.5)
)


for batch_index in selected_indices:
    water_np = (
        assembled.water_spectra[
            batch_index
        ]
        .detach()
        .cpu()
        .numpy()
    )

    mixture_np = (
        assembled.input_spectra[
            batch_index
        ]
        .detach()
        .cpu()
        .numpy()
    )

    snr_value = float(
        simulated_noise.snr[
            batch_index
        ].item()
    )

    plt.figure(
        figsize=(12, 5)
    )

    plt.plot(
        ppm_axis[visible_mask],
        np.real(
            water_np[visible_mask]
        ),
        label="Water",
    )

    plt.plot(
        ppm_axis[visible_mask],
        np.real(
            mixture_np[visible_mask]
        ),
        label="Network input",
        alpha=0.8,
    )

    plt.xlim(
        7.5,
        0.0,
    )

    plt.xlabel(
        "Chemical shift [ppm]"
    )

    plt.ylabel(
        "Real signal"
    )

    plt.title(
        f"Simulated spectrum {batch_index}, "
        f"SNR = {snr_value:.2f}"
    )

    plt.legend()

    plt.grid(
        alpha=0.3
    )

    plt.tight_layout()

    plt.show()